# efficiency improvement cement retrofit NPV simulation

Run the efficiency improvement cement retrofit Monte Carlo simulation and visualize the resulting NPV distribution.

The summary also reports how many simulations have non-negative NPV (NPV >= 0) and how many have negative NPV.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cement.cement_npv_monte_carlo import (
    DEFAULT_RANDOM_SEED,
    DEFAULT_RETROFIT_BAU_MODE,
    DEFAULT_SAMPLE_SIZE,
    simulate_cement_results,
)

from npv_summary import summarize_metric_signs


In [2]:
TECHNOLOGY = 'efficiency_improvement'
SAMPLE_SIZE = DEFAULT_SAMPLE_SIZE
RANDOM_SEED = DEFAULT_RANDOM_SEED
RETROFIT_BAU_MODE = DEFAULT_RETROFIT_BAU_MODE

results_by_technology = simulate_cement_results(
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
    technologies=(TECHNOLOGY,),
    retrofit_bau_mode=RETROFIT_BAU_MODE,
)
simulation = results_by_technology[TECHNOLOGY]
results = pd.DataFrame(simulation)
results.head()


,run_id,technology,technology_type,retrofit_bau_mode,annual_output_t,lifetime_years,capex_eur_per_t,fixed_opex_eur_per_t,variable_opex_eur_per_t,fuel_consumption_mwh_th_per_t,...,bau_variable_opex_eur_per_t,bau_fuel_consumption_mwh_th_per_t,bau_electricity_consumption_mwh_per_t,bau_emissions_tco2_per_t,capex_change_eur_per_t,fixed_opex_change_eur_per_t,variable_opex_change_eur_per_t,fuel_consumption_reduction_fraction,electricity_consumption_reduction_fraction,emissions_reduction_fraction
0,0,efficiency_improvement,retrofit,sampled,1000000.0,25.0,168.624695,14.937909,5.135369,0.651198,...,5.135369,0.657332,0.084504,0.620035,3.145574,0.0,0.0,0.009333,0.060399,0.016369
1,1,efficiency_improvement,retrofit,sampled,1000000.0,25.0,162.956881,14.946918,5.125930,0.589000,...,5.125930,0.649577,0.093378,0.602551,4.179313,0.0,0.0,0.093257,0.082590,0.018236
2,2,efficiency_improvement,retrofit,sampled,1000000.0,25.0,167.911981,14.464250,5.431670,0.640664,...,5.431670,0.697878,0.085678,0.683904,0.740022,0.0,0.0,0.081982,0.059576,0.017513
3,3,efficiency_improvement,retrofit,sampled,1000000.0,25.0,185.081316,14.001728,5.260010,0.670268,...,5.260010,0.693428,0.089539,0.634547,21.133955,0.0,0.0,0.033399,0.183054,0.014023
4,4,efficiency_improvement,retrofit,sampled,1000000.0,25.0,163.659431,14.719678,5.371749,0.638254,...,5.371749,0.694798,0.084480,0.652753,11.775884,0.0,0.0,0.081383,0.064707,0.000436


In [3]:
npv_million_eur = results["npv_eur"] / 1_000_000
lcoc_eur_per_t = results["lcoc_eur_per_t"]
levelized_net_margin_eur_per_t = results["levelized_net_margin_eur_per_t"]

summary = pd.concat(
    [
        npv_million_eur.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "NPV million EUR"
        ),
        lcoc_eur_per_t.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "LCOC EUR/t cement"
        ),
        levelized_net_margin_eur_per_t.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "Levelized net margin EUR/t cement"
        ),
    ],
    axis=1,
)

npv_signs = summarize_metric_signs(npv_million_eur)
npv_sign_summary = pd.DataFrame(
    {
        "NPV category": ["Non-negative (NPV >= 0)", "Negative (NPV < 0)"],
        "Simulation count": [
            npv_signs["non_negative_count"],
            npv_signs["negative_count"],
        ],
        "Simulation share": [
            npv_signs["non_negative_share"],
            1.0 - npv_signs["non_negative_share"],
        ],
    }
)

display(summary)
display(npv_sign_summary)


,NPV million EUR,LCOC EUR/t cement,Levelized net margin EUR/t cement
count,100000.000000,100000.000000,100000.000000
mean,448.668903,107.969245,42.030755
std,43.309555,4.057186,4.057186
min,275.117469,94.096262,25.772669
5%,376.519383,101.366807,35.271876
50%,449.088652,107.929923,42.070077
95%,519.148454,114.728124,48.633193
max,596.759886,124.227331,55.903738


,NPV category,Simulation count,Simulation share
0,Non-negative (NPV >= 0),100000,1.0
1,Negative (NPV < 0),0,0.0


In [4]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(npv_million_eur, bins=50, color="tab:gray", edgecolor="white", alpha=0.8)
ax.axvline(npv_million_eur.mean(), color="tab:blue", linewidth=2, label="Mean")
ax.axvline(npv_million_eur.median(), color="tab:orange", linewidth=2, label="Median")
ax.axvline(0, color="black", linewidth=1, linestyle="--", label="Break-even")

ax.set_title(f"efficiency improvement cement retrofit NPV distribution (n={SAMPLE_SIZE})")
ax.set_xlabel("NPV (million EUR)")
ax.set_ylabel("Frequency")
ax.grid(axis="y", alpha=0.25)
ax.legend()

fig.tight_layout()
plt.show()


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42717/3532807268.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(
    levelized_net_margin_eur_per_t,
    bins=50,
    color="tab:green",
    edgecolor="white",
    alpha=0.8,
)
ax.axvline(
    levelized_net_margin_eur_per_t.mean(),
    color="tab:blue",
    linewidth=2,
    label="Mean",
)
ax.axvline(
    levelized_net_margin_eur_per_t.median(),
    color="tab:orange",
    linewidth=2,
    label="Median",
)
ax.axvline(0, color="black", linewidth=1, linestyle="--", label="Break-even")

ax.set_title(f"efficiency improvement cement retrofit levelized net margin distribution (n={SAMPLE_SIZE})")
ax.set_xlabel("Levelized net margin (EUR/t cement)")
ax.set_ylabel("Frequency")
ax.grid(axis="y", alpha=0.25)
ax.legend()

fig.tight_layout()
plt.show()


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42717/2668037142.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
annual_components = results[
    [
        "annual_revenue_eur",
        "annual_fixed_opex_eur",
        "annual_variable_opex_eur",
        "annual_fuel_cost_eur",
        "annual_electricity_cost_eur",
        "annual_emissions_cost_eur",
        "annual_net_cash_flow_eur",
    ]
] / 1_000_000

annual_components.mean().rename("Mean annual value, million EUR")


annual_revenue_eur             150.000000
annual_fixed_opex_eur           14.332408
annual_variable_opex_eur         5.167223
annual_fuel_cost_eur             7.671399
annual_electricity_cost_eur     14.335492
annual_emissions_cost_eur       50.162518
annual_net_cash_flow_eur        58.330961
Name: Mean annual value, million EUR, dtype: float64

In [7]:
retrofit_columns = [
    "capex_change_eur_per_t",
    "fixed_opex_change_eur_per_t",
    "variable_opex_change_eur_per_t",
    "fuel_consumption_reduction_fraction",
    "electricity_consumption_reduction_fraction",
    "emissions_reduction_fraction",
    "bau_capex_eur_per_t",
    "bau_fixed_opex_eur_per_t",
    "bau_variable_opex_eur_per_t",
    "bau_fuel_consumption_mwh_th_per_t",
    "bau_electricity_consumption_mwh_per_t",
    "bau_emissions_tco2_per_t",
]

available_retrofit_columns = [column for column in retrofit_columns if column in results]
retrofit_summary = results[available_retrofit_columns].describe(
    percentiles=[0.05, 0.5, 0.95]
)
retrofit_summary


,capex_change_eur_per_t,fixed_opex_change_eur_per_t,variable_opex_change_eur_per_t,fuel_consumption_reduction_fraction,electricity_consumption_reduction_fraction,emissions_reduction_fraction,bau_capex_eur_per_t,bau_fixed_opex_eur_per_t,bau_variable_opex_eur_per_t,bau_fuel_consumption_mwh_th_per_t,bau_electricity_consumption_mwh_per_t,bau_emissions_tco2_per_t
count,100000.000000,100000.0,100000.0,100000.000000,1.000000e+05,1.000000e+05,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,13.988549,0.0,0.0,0.049973,9.979483e-02,1.001857e-02,160.012499,14.332408,5.167223,0.666500,0.086635,0.633377
std,8.085978,0.0,0.0,0.028924,5.767616e-02,5.780242e-03,5.770418,0.471200,0.235343,0.040049,0.004713,0.023522
min,0.000110,0.0,0.0,0.000003,7.185888e-07,3.767991e-07,150.000263,13.009529,4.504187,0.610000,0.080000,0.600000
5%,1.405775,0.0,0.0,0.004934,1.006058e-02,9.934831e-04,150.989862,13.451564,4.725440,0.614265,0.080499,0.602579
50%,13.957672,0.0,0.0,0.049872,9.965901e-02,1.003444e-02,160.049191,14.411204,5.207742,0.659535,0.085806,0.629393
95%,26.604482,0.0,0.0,0.095005,1.897232e-01,1.902222e-02,169.007981,14.949464,5.474133,0.741932,0.095517,0.677623
max,27.999954,0.0,0.0,0.100000,1.999993e-01,1.999997e-02,169.999875,14.999998,5.499993,0.779352,0.099946,0.699866
